# 🔎 PCB Quick Ground Truth Inspector

This lightweight notebook allows you to **input any PCB image path** and instantly render the ground truth annotations with:
- 🎨 **Distinct color-coded bounding boxes** (Cyan = Capacitor, Gold = Connector, Green = IC, Magenta = Elec. Cap).
- 🔄 **Automatic Roboflow Taxonomy Aliasing** (Class 1 `Capacitor Jumper` $\rightarrow$ Class 2 `Capacitor`).
- 📊 **Instant Component Counts & Breakdown** (Target components vs non-target components).
- 🔍 **Side-by-Side Comparison**: View **Target 4 Classes** vs **All 23 Raw Annotations**.

In [ ]:
# 1. Setup & Imports
import os
import glob
from pathlib import Path
from collections import Counter
import cv2
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 150
print("✅ Setup completed successfully!")

In [ ]:
# 2. Class Definitions & High-Contrast Colors
RAW_CLASSES = [
    "Button", "Capacitor Jumper", "Capacitor", "Clock", "Connector",
    "Diode", "EM", "Electrolytic Capacitor", "Ferrite Bead", "IC",
    "Inductor", "Jumper", "Led", "Pads", "Pins", "Resistor",
    "Resistor Jumper", "Resistor Network", "Switch", "Test Point",
    "Transistor", "Unknown", "Variable Resistor"
]

# Evaluated Target Classes
EVAL_CLASSES = {
    2: "Capacitor", 
    4: "Connector", 
    7: "Electrolytic Capacitor", 
    9: "IC"
}

# High-Contrast Palette for Clear Visual Differentiation
CLASS_COLORS_RGB = {
    "Capacitor": (0, 220, 255),               # Cyan
    "Capacitor Jumper": (0, 220, 255),        # Cyan (mapped to Capacitor)
    "Connector": (255, 185, 0),               # Gold / Yellow-Orange
    "Electrolytic Capacitor": (255, 50, 180), # Magenta / Hot Pink
    "IC": (0, 255, 100),                      # Bright Neon Green
    "Resistor": (255, 120, 50),               # Orange
    "Resistor Network": (255, 120, 50),       # Orange
    "Resistor Jumper": (255, 120, 50),        # Orange
    "Pins": (180, 80, 255),                   # Violet / Purple
    "Pads": (120, 200, 255),                  # Light Blue
    "Transistor": (255, 230, 80),             # Yellow
    "Diode": (255, 70, 70),                   # Coral Red
    "Default": (180, 180, 180)                # Light Gray
}

def get_color(cname):
    return CLASS_COLORS_RGB.get(cname, CLASS_COLORS_RGB["Default"])

In [ ]:
# 3. Annotation Loader & Box Renderer
def find_label_file(img_path):
    p = Path(img_path)
    # 1. Look for matching labels/ folder
    candidate1 = Path(str(p).replace("/images/", "/labels/").replace("images/", "labels/")).with_suffix(".txt")
    if candidate1.exists():
        return candidate1
    # 2. Look in the same directory
    candidate2 = p.with_suffix(".txt")
    if candidate2.exists():
        return candidate2
    return None

def load_ground_truth(label_path, img_w, img_h, map_capacitor_jumpers=True):
    boxes = []
    if not label_path or not os.path.exists(label_path):
        return boxes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                raw_cid = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = int(round((cx - bw / 2) * img_w))
                y1 = int(round((cy - bh / 2) * img_h))
                x2 = int(round((cx + bw / 2) * img_w))
                y2 = int(round((cy + bh / 2) * img_h))
                
                # Handle Roboflow taxonomy inconsistency: Class 1 -> Class 2
                if map_capacitor_jumpers and raw_cid == 1:
                    cid = 2
                    cname = "Capacitor"
                else:
                    cid = raw_cid
                    cname = EVAL_CLASSES.get(cid, RAW_CLASSES[cid] if cid < len(RAW_CLASSES) else f"Class_{cid}")
                
                boxes.append({
                    "raw_id": raw_cid,
                    "cls_id": cid,
                    "cls_name": cname,
                    "bbox": (x1, y1, x2, y2),
                    "is_target": cid in EVAL_CLASSES or cname in EVAL_CLASSES.values()
                })
    return boxes

def render_boxes(img_rgb, boxes, target_only=True, show_labels=True, box_thickness=2):
    canvas = img_rgb.copy()
    h, w = canvas.shape[:2]
    for b in boxes:
        if target_only and not b["is_target"]:
            continue
        cname = b["cls_name"]
        color = get_color(cname)
        x1, y1, x2, y2 = b["bbox"]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w - 1, x2), min(h - 1, y2)
        
        # Draw rectangle
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, box_thickness)
        
        if show_labels:
            label = f"GT:{cname[:4]}" if target_only else cname
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.38, 1)
            y_label = max(y1 - 4, th + 4)
            cv2.rectangle(canvas, (x1, y_label - th - 3), (x1 + tw + 3, y_label + 3), color, -1)
            cv2.putText(canvas, label, (x1 + 1, y_label), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 0, 0), 1, cv2.LINE_AA)
    return canvas
print("✅ Helper functions loaded!")

---

In [ ]:
# 4. 🎯 Quick Inspection: Enter Any Image Path Below!
# You can change this path to any image in data_samples or elsewhere:
IMAGE_PATH = "data_samples/images/Arty_Bottom_jpg.rf.8b34dbef548b22055e8dae7322d65d13.jpg"
# IMAGE_PATH = "data_samples/images/ATTIOT_Bottom_jpg.rf.8a97ad6664656973c60d95057d9d473c.jpg"

TARGET_ONLY = True              # Set True to show our 4 target classes; False to show all 23 classes
MAP_CAPACITOR_JUMPERS = True    # Set True to map Class 1 (Capacitor Jumper) -> Class 2 (Capacitor)
SHOW_LABELS = True              # Show component text tags above boxes
FIGURE_SIZE = (12, 12)          # Plot display size

# Execution
assert os.path.exists(IMAGE_PATH), f"❌ File not found: {IMAGE_PATH}"
raw_bgr = cv2.imread(IMAGE_PATH)
rgb_img = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
h, w = rgb_img.shape[:2]

lbl_file = find_label_file(IMAGE_PATH)
gt_boxes = load_ground_truth(lbl_file, w, h, map_capacitor_jumpers=MAP_CAPACITOR_JUMPERS)
vis_img = render_boxes(rgb_img, gt_boxes, target_only=TARGET_ONLY, show_labels=SHOW_LABELS)

# Statistics
target_counts = Counter(b["cls_name"] for b in gt_boxes if b["is_target"])
all_counts = Counter(b["cls_name"] for b in gt_boxes)
non_target_counts = Counter(b["cls_name"] for b in gt_boxes if not b["is_target"])

print("=" * 75)
print(f"🖼️ Image: {Path(IMAGE_PATH).name} ({w}x{h} px)")
print(f"📄 Label file: {lbl_file}")
print(f"📦 Total annotations: {len(gt_boxes)} boxes")
print(f"🎯 Target Evaluated Objects ({sum(target_counts.values())} boxes):")
for cname, cnt in sorted(target_counts.items(), key=lambda x: -x[1]):
    print(f"     • {cname:25s}: {cnt:4d} boxes")
if not TARGET_ONLY or non_target_counts:
    print(f"⚪ Non-Target Objects ({sum(non_target_counts.values())} boxes):")
    for cname, cnt in sorted(non_target_counts.items(), key=lambda x: -x[1]):
        print(f"     • {cname:25s}: {cnt:4d} boxes")
print("=" * 75)

# Display
plt.figure(figsize=FIGURE_SIZE)
plt.imshow(vis_img)
mode_title = "Target 4 Classes" if TARGET_ONLY else "All Raw Classes"
plt.title(f"Ground Truth [{mode_title}]: {Path(IMAGE_PATH).name[:45]}\n({sum(target_counts.values())} target objects | {len(gt_boxes)} total in file)", 
          fontsize=12, fontweight='bold', pad=10, color='#1E3A8A')
plt.axis("off")
plt.tight_layout()
plt.show()

---

In [ ]:
# 5. 🔍 Side-by-Side View: Target 4 Classes vs. All 23 Raw Classes
# Use this to see why some components are drawn and where all resistors/pads are!

vis_target = render_boxes(rgb_img, gt_boxes, target_only=True, show_labels=True)
vis_all = render_boxes(rgb_img, gt_boxes, target_only=False, show_labels=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Left: Target 4 Classes
axes[0].imshow(vis_target)
axes[0].set_title(f"[A] Target Evaluated Classes ({sum(target_counts.values())} objects)\nCapacitors (Cyan), Connectors (Gold), ICs (Green)", 
                  fontsize=12, fontweight='bold', color='#065F46', pad=8)
axes[0].axis("off")

# Right: All Raw Classes
axes[1].imshow(vis_all)
axes[1].set_title(f"[B] All Raw Annotations ({len(gt_boxes)} total objects)\nIncluding Resistors (Orange), Pins (Purple), Pads (Blue)", 
                  fontsize=12, fontweight='bold', color='#1E3A8A', pad=8)
axes[1].axis("off")

plt.tight_layout()
plt.show()

---

In [ ]:
# 6. 📋 Browse All 44 Available Test Boards
# Select any sample board by index (0 to 43):
SAMPLE_INDEX = 0

available_samples = sorted(glob.glob("data_samples/images/*.jpg"))
print(f"Found {len(available_samples)} sample test images in data_samples/images/:")
for idx, img_p in enumerate(available_samples):
    mark = "👉 " if idx == SAMPLE_INDEX else "   "
    print(f"{mark}[{idx:2d}] {Path(img_p).name}")

# To view a specific index, set SAMPLE_INDEX above and re-run this block:
chosen_img = available_samples[SAMPLE_INDEX]
print(f"\nCurrently selected sample [{SAMPLE_INDEX}]: {chosen_img}")
# You can now copy-paste this path into Cell 4 or Cell 5 to inspect it!